- 실습 기본 환경 설정


In [ ]:

# 예제 실행 및 시각화를 위한 공통 라이브러리 로딩

# 코랩 환경 등 깃허브 전체를 clone해서 실습하는 경우가 아니라면 
# 공통 라이브러리를 불러오기 위해서 별도의 과정이 필요하므로 code_reference/README.md 파일을 확인하자.
import sys
sys.path.append('../../')

from code_reference import common
from code_reference import visualize as viz

# 시각화 결과를 파일에 저장하지 않음
viz.configure(save_grayscale=False)

# matplotlib 시각화에서 한글 폰트 사용 설정
common.set_korean_plot_env()

# 재현성 보장을 위한 시드 고정
#   여기서 재현성은 '동일 컴퓨터, 동일 버전의 파이썬, 동일 버전의 파이토치' 환경에서 재현이 가능하다는 의미로, 
#   독자의 결과는 저자의 결과와 달라질 수 있다는 점을 밝혀둔다.
#   또한 같은 환경에서도 GPU를 사용하는 경우, 일부 연산의 비결정적 성질로 인해 실행시마다 결과가 조금씩 달라질 수 있다.
SEED = 42
common.set_seed(SEED)

# 실습 환경에 맞는 하드웨어 가속기 장치 객체
device = common.get_device()

# 12-3 양자화로 LLM 가볍게 돌리기

본 노트북은 본문 12-3절의 코드 예제와 관련 내용을 다룬다. 주요 내용은 다음과 같다.
- `BitsAndBytesConfig`로 만드는 양자화 설정 객체와 4비트 양자화 모델 로드
- 양자화 전후의 메모리 사용량과 답변 품질 비교([표 12-9])
- (참고) QLoRA로 함수 호출 작업에 미세 조정하는 전체 과정

> 이번 절 예제는 NVIDIA GPU 환경에서 실습하기를 권장한다. NVIDIA GPU를 사용할 수 없다면 구글 코랩의 무료 티어도 좋은 선택지다.

## 답변 생성 헬퍼

- 12-1절 [코드 12-7]을 모델과 토크나이저를 인자로 받는 함수로 묶었다.
    - 양자화 모델과 비양자화 모델에 같은 방식으로 질문하기 위해서다.

In [ ]:
# 참고 - 답변 생성 헬퍼 (12-1절 [코드 12-7]을 함수로 묶은 것)

import torch

def generate_with_llama(prompt, model, tokenizer, max_new_tokens=256):
    # LLaMA 계열은 종료 토큰이 두 가지다 (<|end_of_text|>, <|eot_id|>).
    terminators = [
        tokenizer.convert_tokens_to_ids('<|end_of_text|>'),
        tokenizer.convert_tokens_to_ids('<|eot_id|>'),
    ]
    messages = [
        {'role': 'system',
         'content': '당신은 한국어를 사용하는 친절한 AI 친구입니다.'},
        {'role': 'user', 'content': prompt},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors='pt', return_dict=False,
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            eos_token_id=terminators,
            do_sample=True, temperature=0.6, top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    llm_generated = output_ids[0][input_ids.shape[-1]:]
    return tokenizer.decode(llm_generated, skip_special_tokens=True)


def gpu_memory_mb():
    """현재 PyTorch가 GPU에 할당한 메모리를 MB 단위로 반환."""
    if not torch.cuda.is_available():
        return 0.0
    return torch.cuda.memory_allocated() / 1024 / 1024

## 4비트 양자화로 모델 불러오기

- bitsandbytes 양자화는 모델을 불러들인 후 따로 적용하는 것이 아니라, 불러오는 단계에서 함께 적용한다.
    - [코드 12-2]의 모델 로드에 양자화 설정 객체 인자(`quantization_config=bnb_config`)만 추가하면 된다.
- 본문 [표 12-7]은 대표적인 PTQ 방식을, [표 12-8]은 모델 파라미터 수별 양자화 추론 메모리 추정치를 정리한다.

In [ ]:
######################################################################################
# 코드 12-13 - 양자화 설정 객체를 만들고 4비트 양자화로 모델 불러오기
######################################################################################

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = 'Bllossom/llama-3.2-Korean-Bllossom-3B'

# 4비트 양자화 설정
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,                      # 4비트로 모델 가중치 로드
    bnb_4bit_quant_type='nf4',              # NF4 형식 사용
    bnb_4bit_compute_dtype=torch.float16,   # 행렬곱 계산은 FP16으로
    bnb_4bit_use_double_quant=True,         # 양자화 상수도 한 번 더 양자화
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model_q = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, quantization_config=bnb_config,
    device_map='auto',                      # 사용 가능한 장치에 자동 배치
)

# 양자화 모델을 불러온 직후의 가속기 메모리를 측정해 두었다가 뒤에서 비교한다
q_memory_mb = gpu_memory_mb()
print(f'양자화 모델 로딩 후 GPU 메모리: {q_memory_mb:.0f} MB')

- `torch.cuda.memory_allocated()`로 메모리를 비교하고, 양자화하지 않은 FP16 모델과 답변 품질도 함께 본다([표 12-9]).
    - 호출할 때마다 다른 답변을 생성하므로, 여러 번 호출해 경향을 확인하는 편이 좋다.

In [ ]:
# 참고 - 양자화(NF4)/비양자화(FP16) 모델의 메모리와 답변 비교
import gc

prompt = '대규모 언어 모델을 적은 자원으로 다루는 방법을 알려 줘.'

print('--- 양자화 적용(NF4) 모델 답변 ---')
print(generate_with_llama(prompt, model_q, tokenizer))

# FP16 비교군 적재를 위해 양자화 모델을 잠시 비운다
del model_q
gc.collect()
torch.cuda.empty_cache()

# 양자화 없이 불러온 비교군 (FP16)
model_fp = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, torch_dtype=torch.float16,
).to(device)
fp_memory_mb = gpu_memory_mb()

print(f'\n양자화 모델 메모리   : {q_memory_mb:.0f} MB')
print(f'비양자화(FP16) 메모리: {fp_memory_mb:.0f} MB')
print('\n--- 양자화 미적용(FP16) 모델 답변 ---')
print(generate_with_llama(prompt, model_fp, tokenizer))

# 비교가 끝나면 FP16 모델은 해제한다 (QLoRA 학습용 메모리 확보)
del model_fp
gc.collect()
torch.cuda.empty_cache()
print(f'\n해제 후 GPU 메모리: {gpu_memory_mb():.0f} MB')

## 참고 - QLoRA로 미세 조정하기

- 최종 원고에서는 QLoRA 실습이 본문에서 빠지고, 저장소의 `docs/qlora.md` 문서로 옮겨졌다.
    - 아래 셀들은 그 실습 과정에 해당하는 코드다. 본문 코드 번호는 부여되지 않는다.
- 풀려는 문제는 사용자 발화를 함수 호출 JSON으로 변환하는 작업이다.
    - 함수 일곱 개를 스키마로 정의하고, 표면 표현 템플릿과 슬롯 풀을 조합해 (사용자 발화, 정답 JSON) 쌍을 합성한다.

In [ ]:
# 참고 - 함수 스키마와 시스템 프롬프트 정의
# 함수 스키마: 모델이 골라야 할 함수 7개를 한 dict로 묶는다.
FUNCTIONS = {
    'create_event': {'desc': '캘린더에 일정 추가',
                     'required': ['date', 'time', 'title']},
    'set_reminder': {'desc': '특정 시각에 알림',
                     'required': ['time', 'message']},
    'get_weather':  {'desc': '도시 날씨 조회',
                     'required': ['city'], 'optional': ['date']},
    'send_message': {'desc': '특정 사람에게 메시지 전송',
                     'required': ['recipient', 'content']},
    'play_music':   {'desc': '음악 재생',
                     'required': ['query']},
    'set_alarm':    {'desc': '알람 설정',
                     'required': ['time'], 'optional': ['label']},
    'search_web':   {'desc': '웹 검색',
                     'required': ['query']},
}


def build_system_prompt():
    """FUNCTIONS dict를 시스템 프롬프트 문자열로 변환한다."""
    lines = [
        '당신은 사용자 요청을 함수 호출 JSON으로 변환하는 어시스턴트입니다.',
        '응답은 반드시 단 하나의 JSON 객체만 출력합니다. '
        '설명·인사·줄바꿈을 포함하지 않습니다.',
        '',
        '사용 가능한 함수:',
    ]
    for name, spec in FUNCTIONS.items():
        args = ', '.join(spec['required'])
        if 'optional' in spec:
            args += ', ' + ', '.join(f'{a}?' for a in spec['optional'])
        lines.append(f'- {name}({args}): {spec["desc"]}')
    lines.append('')
    lines.append('응답 형식: {"function": "함수명", "args": {"key": "value", ...}}')
    return '\n'.join(lines)


SYSTEM_PROMPT = build_system_prompt()
print(SYSTEM_PROMPT)

In [ ]:
# 참고 - 함수 호출 학습·평가 데이터 합성
# 슬롯 풀: 각 인자 자리에 들어갈 후보 값들
TIMES = ['오전 7시', '오전 9시', '오전 10시 30분', '오전 11시',
         '오후 1시', '오후 2시 15분', '오후 3시', '오후 5시',
         '저녁 7시', '저녁 8시 30분', '밤 10시', '밤 11시']
DATES = ['오늘', '내일', '모레', '글피', '이번 주 토요일',
         '다음 주 월요일', '다음 주 금요일', '5월 28일', '6월 3일', '6월 15일']
CITIES = ['서울', '부산', '대구', '인천', '광주',
          '대전', '울산', '제주', '강릉', '전주']
PEOPLE = ['김 부장님', '이 과장님', '박 대리', '최 팀장님',
          '엄마', '아빠', '동생', '수진이', '준호', '지민이']
EVENTS = ['팀 회의', '점심 약속', '치과 예약', '저녁 식사',
          '발표 준비', '운동', '독서 모임', '병원 진료']
MESSAGES_REMIND = ['회의 자료 챙기기', '약 먹기', '우산 챙기기',
                   '카드 결제 확인', '엄마한테 전화하기', '커피 사 오기']
SONGS = ['아이유 신곡', '오아시스의 Wonderwall', '클래식 피아노 음악',
         '잔잔한 재즈', 'BTS Dynamite', '뉴진스 노래', '비 오는 날 음악']
QUERIES = ['파이썬 데코레이터 사용법', '제주도 흑돼지 맛집', '오늘 환율 정보',
           '내일 야구 경기 결과', '서울 카페 추천', '딥러닝 기초 강의',
           '리눅스 셸 명령어']
MSG_CONTENTS = ['오늘 회의 좀 늦을 것 같아', '저녁 같이 먹을래?',
                '내일 일정 변경 가능?', '주말에 시간 어때?',
                '확인 부탁드립니다', '자료 받았습니다 감사합니다']
ALARM_LABELS = ['기상', '약 복용', '회의 알림', '운동 시작', '저녁 약속']

# 학습용 템플릿: 함수당 4종의 표면 표현
TRAIN_TEMPLATES = {
    'create_event': ['{date} {time}에 "{title}" 일정 잡아 줘',
                     '{date} {time} {title} 일정 추가해 줘',
                     '캘린더에 {date} {time}, {title} 등록해 줘',
                     '{title}을 {date} {time}에 일정으로 넣어 줘'],
    'set_reminder': ['{time}에 "{message}" 알림 보내 줘',
                     '{time}에 {message} 잊지 않게 알려 줘',
                     '{message} 알람 {time}에 맞춰 줘',
                     '{time}이 되면 {message} 알려 주세요'],
    'get_weather': ['{city} 날씨 알려 줘', '{date} {city} 날씨 어때?',
                    '{city}의 {date} 날씨가 궁금해', '{date} {city} 비 오나?'],
    'send_message': ['{recipient}한테 "{content}" 라고 메시지 보내 줘',
                     '{recipient}에게 {content} 전해 줘',
                     '{recipient}한테 메시지 보낼래: {content}',
                     '{content}이라고 {recipient}에게 전송해 줘'],
    'play_music': ['{query} 틀어 줘', '{query} 재생해 줘',
                   '{query} 좀 들려줘', '{query} 듣고 싶어'],
    'set_alarm': ['{time}에 알람 맞춰 줘', '{time}에 {label} 알람 설정해 줘',
                  '내일 {time} 알람 부탁해', '{time}에 {label} 깨워 줘'],
    'search_web': ['{query} 검색해 줘', '{query} 찾아 줘',
                   '{query} 정보 알아봐 줘', '{query}에 대해 검색해 봐'],
}

# 평가용 템플릿: 학습과 겹치지 않는 별개 표현
EVAL_TEMPLATES = {
    'create_event': ['{date} {time}부터 {title} 잡아 둘래?',
                     '이번에 {date} {time}에 {title} 있는데 캘린더 부탁해'],
    'set_reminder': ['{time}이 되면 {message} 알리도록 해 줘',
                     '나 {time}에 {message} 까먹지 말게 좀 챙겨 줘'],
    'get_weather': ['{city} 오늘 우산 필요한가?',
                    '{date} {city} 기온 알려 주실래요?'],
    'send_message': ['{recipient}한테 "{content}" 보내려고 하는데',
                     '{recipient}에게 "{content}" 라고 카톡 좀'],
    'play_music': ['{query} 좀 들어 보고 싶은데',
                   '지금 분위기에 {query} 어울릴 듯'],
    'set_alarm': ['{time}에 일어나야 해 알람 좀', '{time} {label}로 알람 부탁'],
    'search_web': ['{query} 한 번 찾아 봐 줄래?', '{query} 관련해서 좀 알아봐'],
}


def gen_sample(func_name, templates):
    """함수 이름과 템플릿을 받아 (USER 발화, GOLD JSON dict) 쌍을 반환."""
    tpl = random.choice(templates[func_name])
    if func_name == 'create_event':
        date, time, title = (random.choice(DATES), random.choice(TIMES),
                             random.choice(EVENTS))
        user = tpl.format(date=date, time=time, title=title)
        args = {'date': date, 'time': time, 'title': title}
    elif func_name == 'set_reminder':
        time, msg = random.choice(TIMES), random.choice(MESSAGES_REMIND)
        user = tpl.format(time=time, message=msg)
        args = {'time': time, 'message': msg}
    elif func_name == 'get_weather':
        city = random.choice(CITIES)
        date = random.choice(DATES) if '{date}' in tpl else None
        user = tpl.format(city=city, date=date or '')
        args = {'city': city}
        if date is not None:
            args['date'] = date
    elif func_name == 'send_message':
        recipient, content = random.choice(PEOPLE), random.choice(MSG_CONTENTS)
        user = tpl.format(recipient=recipient, content=content)
        args = {'recipient': recipient, 'content': content}
    elif func_name == 'play_music':
        query = random.choice(SONGS)
        user = tpl.format(query=query)
        args = {'query': query}
    elif func_name == 'set_alarm':
        time = random.choice(TIMES)
        label = random.choice(ALARM_LABELS) if '{label}' in tpl else None
        user = tpl.format(time=time, label=label or '')
        args = {'time': time}
        if label is not None:
            args['label'] = label
    elif func_name == 'search_web':
        query = random.choice(QUERIES)
        user = tpl.format(query=query)
        args = {'query': query}
    return user, {'function': func_name, 'args': args}


def make_dataset(templates, n_per_func):
    """함수별 n_per_func건씩 생성해 셔플한 list를 반환."""
    samples = []
    for fname in FUNCTIONS:
        for _ in range(n_per_func):
            user, gold = gen_sample(fname, templates)
            samples.append({'user': user,
                            'assistant': json.dumps(gold, ensure_ascii=False)})
    random.shuffle(samples)
    return samples


train_samples = make_dataset(TRAIN_TEMPLATES, n_per_func=22)
eval_samples = make_dataset(EVAL_TEMPLATES, n_per_func=3)
print(f'학습 샘플: {len(train_samples)}건 / 평가 샘플: {len(eval_samples)}건')
for s in train_samples[:2]:
    print(f'  USER : {s["user"]}')
    print(f'  GOLD : {s["assistant"]}')

- 평가용 추론은 그리디 디코딩(`do_sample=False`)을 사용해 어댑터 OFF/ON 비교에서 생성의 무작위성을 배제한다.

In [ ]:
# 참고 - 함수 호출 추론 헬퍼와 채점 함수
TERMINATORS = [
    tokenizer.convert_tokens_to_ids('<|end_of_text|>'),
    tokenizer.convert_tokens_to_ids('<|eot_id|>'),
]
MAX_INPUT_LEN = 512


def generate_funccall(user_text, model, max_new_tokens=128):
    """함수 호출 평가 전용 추론 헬퍼 (그리디 디코딩)."""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_text},
    ]
    input_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors='pt', return_dict=False,
        truncation=True, max_length=MAX_INPUT_LEN,
    ).to(model.device)
    with torch.no_grad():
        output_ids = model.generate(
            input_ids, max_new_tokens=max_new_tokens,
            eos_token_id=TERMINATORS, do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    return tokenizer.decode(
        output_ids[0][input_ids.shape[-1]:], skip_special_tokens=True,
    ).strip()


def try_extract_json(text):
    """모델 출력에서 가장 바깥의 JSON 객체 한 개를 뽑는다."""
    m = re.search(r'\{.*\}', text, re.DOTALL)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except json.JSONDecodeError:
        return None


def score_one(pred_text, gold):
    """한 샘플 채점: parse / func_ok / args_ok."""
    pred = try_extract_json(pred_text)
    gold_obj = json.loads(gold)
    if pred is None:
        return {'parse': False, 'func_ok': False, 'args_ok': False}
    func_ok = pred.get('function') == gold_obj['function']
    args_ok = func_ok and pred.get('args') == gold_obj['args']
    return {'parse': True, 'func_ok': func_ok, 'args_ok': args_ok}


def evaluate(model, samples, label):
    """여러 샘플을 순회하며 세 지표를 합산하고 (rows, stats)를 반환."""
    stats = {'parse': 0, 'func_ok': 0, 'args_ok': 0}
    rows = []
    for s in samples:
        out = generate_funccall(s['user'], model)
        sc = score_one(out, s['assistant'])
        for k in stats:
            stats[k] += int(sc[k])
        rows.append((s['user'], s['assistant'], out, sc))
    n = len(samples)
    print(f'[{label}] n={n} '
          f'JSON parse {stats["parse"]}/{n}, '
          f'function {stats["func_ok"]}/{n}, '
          f'args 완전일치 {stats["args_ok"]}/{n}')
    return rows, stats

- 시스템 프롬프트와 사용자 발화는 입력 부분, 정답 JSON은 정답 부분이다.
    - 디코더 전용 모델이므로 둘을 이어 붙인 뒤 입력 부분의 손실을 무시하도록 레이블을 가공한다.

In [ ]:
# 참고 - 함수 호출 합성 데이터를 LLaMA 입력 형식으로 가공
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq

MAX_TARGET_LEN = 160

# LLaMA 토크나이저는 left 패딩이 기본(생성에 적합). 학습에는 right 패딩을 쓴다.
tokenizer.padding_side = 'right'
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


def tokenize_example(ex):
    """한 샘플 -> {'input_ids', 'labels', 'attention_mask'}."""
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': ex['user']},
    ]
    prompt_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors=None, return_dict=False,
        truncation=True, max_length=MAX_INPUT_LEN,
    )
    target_text = ex['assistant'] + tokenizer.eos_token
    target_ids = tokenizer(
        target_text, add_special_tokens=False,
        truncation=True, max_length=MAX_TARGET_LEN,
    )['input_ids']
    input_ids = prompt_ids + target_ids
    # 입력 토큰 자리는 -100으로 마스킹 -> 손실에서 제외, 정답 토큰만 학습.
    labels = [-100] * len(prompt_ids) + target_ids
    return {'input_ids': input_ids, 'labels': labels,
            'attention_mask': [1] * len(input_ids)}


tokenized_train = Dataset.from_list(train_samples).map(
    tokenize_example, remove_columns=['user', 'assistant'],
)
print(f'토크나이즈 완료. 첫 샘플 길이: '
      f'{len(tokenized_train[0]["input_ids"])} 토큰')

# DataCollatorForSeq2Seq: 가변 길이 input_ids·labels를 함께 패딩.
# (DataCollatorForLanguageModeling은 labels를 input_ids로 덮어써 -100과 충돌)
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, padding='longest', label_pad_token_id=-100,
)

- QLoRA는 네 단계로 준비한다.
    1. 4비트 양자화로 사전 학습 모델을 불러온다.
    2. 양자화 모델을 학습 가능한 상태로 전환한다(`prepare_model_for_kbit_training()`).
    3. LoRA 어댑터를 추가한다(`get_peft_model()`).
    4. 학습 대상 파라미터를 확인한다.

In [ ]:
# 참고 - QLoRA 모델 준비
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# (1) 4비트 양자화로 사전 학습 모델 로드
model = AutoModelForCausalLM.from_pretrained(
    'Bllossom/llama-3.2-Korean-Bllossom-3B',
    quantization_config=bnb_config,         # [코드 12-14] 참조
    device_map='auto',
)
# (2) 양자화 모델을 학습 가능 상태로 전환
model = prepare_model_for_kbit_training(model)

# (3) LoRA 어댑터 계층 추가
lora_config = LoraConfig(                   # LoRA 설정 객체 생성
    r=16,                                   # 저랭크 분해 차원
    lora_alpha=32,                          # LoRA 스케일 계수 (alpha / r)
    target_modules=['q_proj', 'v_proj'],    # 어텐션의 Q, V 선형 계층에만 적용
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',                  # 인과언어모델 (LLaMA, GPT 등)
)
# LoRA 설정 객체를 사용해 모델에 LoRA 어댑터 계층 추가
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

- LLaMA 계열은 디코더 전용 구조이므로 `Seq2SeqTrainer`가 아니라 `Trainer`를 사용한다.

In [ ]:
# 참고 - QLoRA 학습 엔진 생성 및 학습 실행
from transformers import TrainingArguments, Trainer

# 학습 시에는 인과 마스크가 자연스럽게 동작하도록 right 패딩을 사용한다.
tokenizer.padding_side = 'right'

training_args = TrainingArguments(          # 학습 설정 객체
    output_dir='../../checkpoint/qlora-funccall',
    num_train_epochs=3,                     # 154건짜리 작은 데이터라 3 에포크
    per_device_train_batch_size=2,          # 가속기 장치별 배치 크기 2
    gradient_accumulation_steps=4,          # 4 배치마다 1회 최적화 (실효 배치 8)
    learning_rate=2e-4,                     # LoRA, QLoRA 권장 수준
    fp16=True,                              # LoRA 어댑터, 옵티마이저 연산 자료형
    save_strategy='no',
    report_to='none',
    logging_steps=5,                        # 학습 로그 출력 빈도
    seed=42,
)

trainer = Trainer(
    model=model, args=training_args,
    train_dataset=tokenized_train,          # 함수 호출용 합성 데이터셋
    data_collator=data_collator,
)
# Trainer가 네이티브 학습 로그를 직접 출력한다. 직접 에포크 print는 만들지 않고,
# 반환된 metrics의 train_runtime으로 전체 학습 시간만 사람이 읽기 좋게 출력한다(§7.7).
result = trainer.train()
print(f'전체 학습 시간: {result.metrics["train_runtime"]:.1f}초')

# 학습이 끝나면 어댑터만 저장한다 (사전 학습 가중치는 저장하지 않는다)
model.save_pretrained('../../checkpoint/qlora-funccall/adapter')
# 다시 불러올 때는 PeftModel.from_pretrained(base_model, '어댑터 경로') 사용

In [ ]:
# 참고 - log_history에서 훈련 손실을 뽑아 학습 곡선을 그린다 (§7.7)
train_curve = [(rec['step'], rec['loss'])
               for rec in trainer.state.log_history if 'loss' in rec]

viz.plot_history(
    train_curve, label='훈련 손실',
    title='QLoRA 함수 호출 학습 곡선', x_label='스텝', y_label='손실',
)

- `model.disable_adapter()` 컨텍스트에서는 어댑터가 비활성화되어, 미세 조정 전 양자화 모델의 추론 결과를 얻을 수 있다.
    - 같은 모델 객체로 어댑터 ON/OFF를 비교할 수 있다는 것이 LoRA의 장점이다.

In [ ]:
# 참고 - LoRA 어댑터 계층 활성화/비활성화별 모델 평가
model.eval()

# 어댑터 계층 비활성화 (미세 조정 전 양자화 모델 추론)
print('### 어댑터 계층 비활성화')
with model.disable_adapter():
    off_rows, off_stats = evaluate(model, eval_samples, '어댑터 OFF')

# 어댑터 계층 활성화 (QLoRA 미세 조정 적용 양자화 모델 추론)
print('### 어댑터 계층 활성화')
on_rows, on_stats = evaluate(model, eval_samples, '어댑터 ON')

In [ ]:
# 참고 - 앞 3건을 OFF vs ON 으로 나란히 출력
for i in range(3):
    user, gold, _, _ = on_rows[i]
    print(f'\nUSER : {user}')
    print(f'GOLD : {gold}')
    print(f'OFF  : {off_rows[i][2][:200]}')
    print(f'ON   : {on_rows[i][2][:200]}')

## 정리

- 양자화는 모델을 불러오는 단계에서 함께 적용한다. `Auto` 모델 클래스에 양자화 설정 객체만 넘기면 된다.
- 4비트 양자화는 메모리를 크게 줄이지만 품질 손실이 따른다. 정밀한 답변이 필요한 작업에서는 8비트를 쓰거나 양자화하지 않는 편이 안전하다.
- LoRA 어댑터는 사전 학습 파라미터를 고정한 채 작은 행렬만 학습하므로, 양자화 모델에도 적용할 수 있다(QLoRA).